# Build an Octuple tokenizer + tokenized dataset (chunked `.pt`)

This is the **Octuple** counterpart to `build-remi-dataset.ipynb`. It:

1. builds an **Octuple** tokenizer config tuned for long-form generation and continuation,
2. saves `Compose_Octuple.json` (**no BPE** — see below),
3. tokenizes the corpus **in parallel**, splitting long pieces at **bar boundaries** and **re-basing bar indices** so every piece starts at `Bar_0`,
4. writes `chunk_0000.pt … chunk_NNNN.pt` as `list[np.ndarray]` where each array is `(num_notes, 8)` `int16`,
5. writes `manifest.json` with per-field vocab sizes, note counts and length statistics.

## What changes versus REMI, and why it matters

REMI is a *single* stream of ids: one id per pitch, one per velocity, one per duration, one per position. Octuple is a **multi-vocabulary** tokenizer — one step per **note**, and each step is a tuple of 8 ids drawn from 8 independent vocabularies:

```
(Pitch, Position, Bar, Velocity, Duration, Program, Tempo, TimeSig)
```

(That is the order miditok 3.0.6 actually emits — not the order the field names are usually listed in. It has moved between releases, so nothing below hardcodes it; section 2 detects the columns by inspecting each field's vocabulary.)

Three consequences drive every design decision below.

**1. No BPE.** BPE merges adjacent ids in a flat stream; there is no flat stream here. `miditok` refuses to train BPE on a multi-vocabulary tokenizer, so the `TRAIN_BPE` machinery from the REMI notebook is gone. You lose the 2–3× compression BPE gave you — and get roughly 4–8× back for free, because one Octuple step carries what REMI spent 4–8 tokens on. Net: an Octuple sequence for the same music is **substantially shorter** than a BPE'd REMI sequence, which is exactly why this is the right tokenization for long-form.

**2. There is no single `vocab_size`.** There are eight, and the model needs all of them. `VOCAB_SIZES` is written into the manifest and read by the training notebook. The `< 32768` assertion still holds per field (comfortably — the largest field is Bar), so window storage stays `int16`.

**3. `Bar` is an absolute index, not a marker.** In REMI, `Bar_None` is a delimiter you can cut on anywhere. In Octuple, bar 137 of a piece is literally the token `Bar_137`, and the Bar vocabulary is finite (`MAX_BAR_EMBEDDING`). Two things follow, and both are handled in section 5: long pieces must be **split** before they run off the end of the Bar vocabulary, and every split piece must be **re-based** to start at `Bar_0` — otherwise half your training windows begin at `Bar_400` and the model learns that bar indices are meaningless noise. The same trick reappears at generation time to let a model with a 256-bar vocabulary write a song of unbounded length.

**Runtime:** roughly 1–3 h on a Kaggle CPU session for ~150k MIDI files. It is **resumable** — existing chunks are skipped, so a timed-out session picks up where it stopped.

*Written against miditok 3.0.x / symusic 0.6.x.*

In [1]:
%pip install -q "miditok>=3.0" symusic tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.0/159.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 39.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

**`MODE`** — `per_track` (default) encodes each instrument track as its own sequence. Octuple always carries a `Program` field, so a `multi_instrument` stream is a first-class option here in a way it never quite was in REMI: the model can learn full arrangements and you still know which instrument every note belongs to. Pick based on the product:

- `per_track` — single-line phrasing and continuation. Matches a user playing one instrument.
- `multi_instrument` — full-band arrangement generation. Sequences are longer (all tracks interleaved by time) and the `Program` head actually has work to do.

**`MAX_BARS_PER_PIECE` / `MAX_BAR_EMBEDDING`** are the pair that keeps Bar ids in range. Keep `MAX_BARS_PER_PIECE ≤ MAX_BAR_EMBEDDING`. 256 bars is about 8 minutes at 120 bpm in 4/4 — long enough that a training window is a genuine song section, short enough that the Bar embedding table stays small. `MAX_BAR_EMBEDDING` is set higher (1024) because tokenization happens *before* splitting, so the tokenizer must be able to represent bar 900 of a long piece even though no stored piece will contain it.

**`MIN_SEQ_NOTES`** should equal your training `SEQ_LEN + 1`, now measured in **notes**, not tokens. A 1024-note sequence is a real piece of music — a few minutes of a busy track, longer for a sparse one. Section 6 reports how much of the corpus survives this filter; watch it, because the Octuple minimum bites harder than the REMI one did.

**`MAX_NOTES_PER_PIECE`** is a safety valve for pathologically dense files (a 20-voice orchestral reduction can put 4000 notes in 32 bars). It caps a piece regardless of bar count, still cutting on a bar line.

In [2]:
import os, json, math, random, time
from pathlib import Path

CONFIG = {
    # ---- paths: EDIT THESE ----
    "MIDI_DIRS": [
        # "/kaggle/input/godzilla-midi-subset",     # any number of folders, searched recursively
        "/kaggle/input/datasets/perryplay/godzillak150",
    ],
    "OUT_DIR":        "/kaggle/working/octuple_dataset",
    "TOKENIZER_NAME": "Compose_Octuple.json",

    # ---- corpus scope ----
    "MODE": "per_track",          # "per_track" | "multi_instrument"
    "MAX_FILES": None,            # None = all
    "DROP_DRUMS": True,
    "SEED": 42,

    # ---- Octuple tokenizer ----
    "BEAT_RES": {(0, 4): 8, (4, 12): 4},   # 32nd grid for 4 beats, 16th beyond
    "NUM_VELOCITIES": 24,
    "USE_TEMPOS": True,
    "NUM_TEMPOS": 32,
    "TEMPO_RANGE": (50, 200),
    "USE_TIME_SIGNATURES": True,
    "PITCH_RANGE": (21, 109),
    "MAX_BAR_EMBEDDING": 1024,    # size of the Bar field vocabulary
    # Octuple does NOT support chords, rests, sustain pedals or pitch bends.
    # There is no BPE: multi-vocabulary tokenizers cannot be BPE-trained.

    # ---- output ----
    "MIN_SEQ_NOTES":       1025,   # = training SEQ_LEN + 1, measured in NOTES
    "MAX_BARS_PER_PIECE":  256,    # split longer streams on a bar line, then re-base to Bar_0
    "MAX_NOTES_PER_PIECE": 4096,   # hard cap for very dense pieces
    "FILES_PER_CHUNK": 12000,
    "N_WORKERS": max(1, (os.cpu_count() or 4) - 1),
}

assert CONFIG["MAX_BARS_PER_PIECE"] <= CONFIG["MAX_BAR_EMBEDDING"], \
    "a stored piece could contain a Bar id the tokenizer cannot represent"

random.seed(CONFIG["SEED"])
OUT = Path(CONFIG["OUT_DIR"]); OUT.mkdir(parents=True, exist_ok=True)
CHUNK_DIR = OUT / "chunks"; CHUNK_DIR.mkdir(exist_ok=True)
TOKENIZER_PATH = OUT / CONFIG["TOKENIZER_NAME"]
print("output ->", OUT, "| workers:", CONFIG["N_WORKERS"])

output -> /kaggle/working/octuple_dataset | workers: 3


## 2. Build the Octuple tokenizer

Two things to verify in the printout before continuing:

- **eight fields**, with the `Bar` field roughly `MAX_BAR_EMBEDDING` entries long,
- **`one_token_stream: True`**. Octuple forces `use_programs=True` and emits a single interleaved stream per `Score`. That is why `per_track` mode below encodes one track at a time rather than relying on the tokenizer to split them — feeding it a multi-track score returns *one* sequence with a varying `Program` field, not one sequence per track.

`FIELDS` is derived by inspecting each field's vocabulary rather than assuming miditok's ordering. Field order has shifted between miditok releases, and every downstream step (bar splitting, re-basing, per-field loss weights, constrained sampling) indexes by name.

In [3]:
from miditok import Octuple, TokenizerConfig, TokSequence
import inspect, warnings

# miditok warns on EVERY Octuple construction that attribute controls are
# incompatible with multi-vocabulary tokenizers. It is unconditional - there is no
# config that avoids it - and it is irrelevant here, so it is silenced once,
# globally. The worker module below does the same in each subprocess.
warnings.filterwarnings("ignore", message=".*Attribute controls are not compatible.*")

_kw = dict(
    beat_res=CONFIG["BEAT_RES"],
    num_velocities=CONFIG["NUM_VELOCITIES"],
    use_chords=False,             # unsupported by Octuple
    use_rests=False,              # unsupported by Octuple
    use_tempos=CONFIG["USE_TEMPOS"],
    num_tempos=CONFIG["NUM_TEMPOS"],
    tempo_range=CONFIG["TEMPO_RANGE"],
    use_time_signatures=CONFIG["USE_TIME_SIGNATURES"],
    use_sustain_pedals=False,
    use_pitch_bends=False,
    pitch_range=CONFIG["PITCH_RANGE"],
    use_programs=True,            # forced by Octuple anyway
    max_bar_embedding=CONFIG["MAX_BAR_EMBEDDING"],
)
_accepted = set(inspect.signature(TokenizerConfig.__init__).parameters)
_extra = {}
if "max_bar_embedding" not in _accepted:
    # newer miditok takes it through **kwargs into additional_params; older took it positionally
    _extra["max_bar_embedding"] = _kw.pop("max_bar_embedding")
_dropped = [k for k in _kw if k not in _accepted and k != "max_bar_embedding"]
if _dropped:
    print("note: this miditok version ignores", _dropped)

cfg = TokenizerConfig(**{k: v for k, v in _kw.items() if k in _accepted}, **_extra)
cfg.additional_params.setdefault("max_bar_embedding", CONFIG["MAX_BAR_EMBEDDING"])
tokenizer = Octuple(cfg)

assert getattr(tokenizer, "is_multi_voc", False), "expected a multi-vocabulary tokenizer"


def field_vocabs(tk):
    """Return the per-field vocab dicts as a list, whatever shape miditok used."""
    v = tk.vocab
    assert isinstance(v, list) and isinstance(v[0], dict), f"unexpected vocab layout: {type(v)}"
    return v


def detect_fields(vocabs):
    """Map field name -> index by looking for its token prefix. Never assume order."""
    names = ["Pitch", "Velocity", "Duration", "Position", "Bar",
             "Program", "Tempo", "TimeSig"]
    out = {}
    for i, voc in enumerate(vocabs):
        for n in names:
            if n in out:
                continue
            if any(str(t).startswith(n + "_") for t in voc):
                out[n] = i
                break
    return out


VOCABS      = field_vocabs(tokenizer)
VOCAB_SIZES = [len(v) for v in VOCABS]
FIELDS      = detect_fields(VOCABS)
NUM_FIELDS  = len(VOCABS)

print("fields:", NUM_FIELDS, "| one_token_stream:", getattr(tokenizer, "one_token_stream", None))
for name, i in sorted(FIELDS.items(), key=lambda kv: kv[1]):
    print(f"  [{i}] {name:9s} vocab {VOCAB_SIZES[i]:5d}")
assert "Bar" in FIELDS and "Position" in FIELDS, "could not locate the Bar/Position fields"
assert max(VOCAB_SIZES) < 32768, "a field vocab >= 32768 breaks int16 storage"

fields: 8 | one_token_stream: True
  [0] Pitch     vocab   155
  [1] Position  vocab   100
  [2] Bar       vocab  1028
  [3] Velocity  vocab    28
  [4] Duration  vocab    68
  [5] Program   vocab   133
  [6] Tempo     vocab    36
  [7] TimeSig   vocab    13


## 3. Find the MIDI files

Identical to the REMI notebook. `MAX_FILES` subsamples **randomly** rather than truncating, because MIDI corpora are laid out alphabetically or by source archive and `files[:150000]` would hand you a corpus that is overwhelmingly one genre or one contributor.

In [4]:
exts = ("*.mid", "*.midi", "*.MID")
files = []
for d in CONFIG["MIDI_DIRS"]:
    p = Path(d)
    if not p.exists():
        print("WARNING: missing dir", d); continue
    for e in exts:
        files.extend(p.rglob(e))
files = sorted({str(f) for f in files})
print(f"{len(files):,} MIDI files found")
assert files, "No MIDI files - check CONFIG['MIDI_DIRS']"

if CONFIG["MAX_FILES"] and len(files) > CONFIG["MAX_FILES"]:
    files = sorted(random.sample(files, CONFIG["MAX_FILES"]))
    print(f"randomly subsampled to {len(files):,}")

150,000 MIDI files found


## 4. Save the tokenizer

No BPE stage — `miditok` raises on `train()` for multi-vocabulary tokenizers, and the merge idea does not transfer: there is no adjacency to merge over when every step is already a full note.

The tokenizer is saved **before** the corpus pass so a crash during tokenization never costs you anything, and so the workers can each rebuild it from JSON instead of receiving it by pickle.

One subtlety worth knowing about: some miditok versions **grow the Bar vocabulary on the fly** when they meet a piece longer than `max_bar_embedding`. That would make a worker's vocabulary disagree with the saved JSON, and training would then hit ids past the end of its embedding table. The workers are therefore given the frozen `VOCAB_SIZES` and drop any piece containing an out-of-range id, with the count reported in section 6.

In [5]:
def save_tokenizer(tk, path):
    """miditok renamed save_params -> save in 3.0.5+; support both."""
    last = None
    for meth in ("save", "save_params"):
        if hasattr(tk, meth):
            try:
                getattr(tk, meth)(Path(path)); return Path(path)
            except Exception as e:
                last = e
    raise RuntimeError(f"could not save tokenizer: {last}")

save_tokenizer(tokenizer, TOKENIZER_PATH)
print("saved ->", TOKENIZER_PATH)

_re = Octuple(params=str(TOKENIZER_PATH))     # the exact call the training notebook makes
_re_sizes = [len(v) for v in field_vocabs(_re)]
print("reload OK | vocab sizes:", _re_sizes)
assert _re_sizes == VOCAB_SIZES, "reloaded tokenizer disagrees with the in-memory one"

saved -> /kaggle/working/octuple_dataset/Compose_Octuple.json
reload OK | vocab sizes: [155, 100, 1028, 28, 68, 133, 36, 13]


## 5. Tokenize the corpus in parallel

Per file the worker does five things, in an order that matters:

1. **Load with symusic, drop drum tracks.** Channel-10 tracks would otherwise teach the model to answer a guitar riff with a hi-hat pattern.
2. **Encode.** In `per_track` mode each track is encoded as its own single-track `Score`, because Octuple's `one_token_stream` behaviour would otherwise interleave everything into one sequence.
3. **Reshape to `(num_notes, 8)`.** `TokSequence.ids` for a multi-vocabulary tokenizer is a list of 8-element lists; `np.asarray` turns it into the matrix the training notebook wants.
4. **Split on bar lines** at `MAX_BARS_PER_PIECE` bars or `MAX_NOTES_PER_PIECE` notes, whichever comes first. The Bar column is non-decreasing, so `searchsorted` finds the cut in log time — no accumulation pass like TSD needed, and no token scan like REMI needed.
5. **Re-base the Bar column** so each piece starts at `Bar_0`, then range-check every field and keep pieces of at least `MIN_SEQ_NOTES`.

Step 5 is the one with no REMI equivalent and the one that will quietly ruin the run if skipped. A window sliced out of the middle of a long piece would otherwise carry bar indices in the 300s, so the model would see `Bar_312` exactly as often as any other high bar number — a few dozen times across the whole corpus — and learn nothing from the field. After re-basing, low bar numbers are dense and the model learns what it should: how far into a section it is, and that the number advances by one.

In [6]:
import numpy as np, torch, importlib, sys
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm

# The worker is written to a real module file rather than defined in this cell.
# ProcessPoolExecutor pickles the function *by name*, so a cell-defined function
# fails under the "spawn" start method (and inside some notebook runners). An
# importable module works everywhere.
WORKER_PY = OUT / "octuple_worker.py"
WORKER_PY.write_text('''
import warnings
import numpy as np

# Silenced at import time so it applies to every Octuple built in this process.
warnings.filterwarnings("ignore", message=".*Attribute controls are not compatible.*")

_W = {}

def init_worker(tok_path, mode, drop_drums, min_notes, max_bars, max_notes,
                bar_idx, bar0_id, vocab_sizes):
    from miditok import Octuple
    tk = Octuple(params=tok_path)
    _W.update(tk=tk, mode=mode, drop_drums=drop_drums, min_notes=min_notes,
              max_bars=max_bars, max_notes=max_notes, bar_idx=bar_idx,
              bar0_id=bar0_id, vocab=np.asarray(vocab_sizes, dtype=np.int64))

def split_on_bars(arr, bar_idx, bar0_id, max_bars, max_notes):
    # cut into pieces of <= max_bars bars / <= max_notes notes, always on a bar line
    n = len(arr)
    bars = arr[:, bar_idx].astype(np.int32) - bar0_id
    monotonic = bool(np.all(np.diff(bars) >= 0))
    if not monotonic:
        return [arr[i:i + max_notes] for i in range(0, n, max_notes)]
    out, start = [], 0
    while start < n:
        by_bar = int(np.searchsorted(bars, bars[start] + max_bars, side="left"))
        end = by_bar if by_bar > start else n
        if end - start > max_notes:
            cut = start + max_notes
            # snap back to the start of the bar we landed inside
            snap = int(np.searchsorted(bars, bars[cut], side="left"))
            end = snap if snap > start else cut
        end = min(end, n)
        out.append(arr[start:end])
        start = end
    return out

def rebase_bars(piece, bar_idx, bar0_id):
    # shift the Bar column so the piece starts at Bar_0
    col = piece[:, bar_idx].astype(np.int32)
    shift = int(col[0]) - bar0_id
    if shift == 0:
        return piece
    piece = piece.copy()
    piece[:, bar_idx] = (col - shift).astype(piece.dtype)
    return piece

def _encode(tk, score):
    res = tk.encode(score)
    seqs = res if isinstance(res, list) else [res]
    out = []
    for s in seqs:
        ids = getattr(s, "ids", None)
        if not ids:
            continue
        a = np.asarray(ids, dtype=np.int32)
        if a.ndim == 2 and a.shape[0] > 0:
            out.append(a)
    return out

def tokenize_one(path):
    from symusic import Score
    tk = _W["tk"]
    try:
        score = Score(str(path))
    except Exception:
        return []
    tracks = [t for t in score.tracks if not (_W["drop_drums"] and t.is_drum)]
    if not tracks:
        return []

    raw = []
    try:
        if _W["mode"] == "per_track":
            for t in tracks:
                one = score.copy()
                one.tracks = [t]
                raw.extend(_encode(tk, one))
        else:
            score.tracks = tracks
            raw.extend(_encode(tk, score))
    except Exception:
        return []

    bar_idx, bar0 = _W["bar_idx"], _W["bar0_id"]
    vocab, out = _W["vocab"], []
    for arr in raw:
        if arr.shape[1] != len(vocab):
            continue
        for piece in split_on_bars(arr, bar_idx, bar0, _W["max_bars"], _W["max_notes"]):
            if len(piece) < _W["min_notes"]:
                continue
            piece = rebase_bars(piece, bar_idx, bar0)
            if piece.min() < 0 or bool(np.any(piece.max(axis=0) >= vocab)):
                continue                      # out-of-range id: drop rather than corrupt
            out.append(piece.astype(np.int16))
    return out
''')

sys.path.insert(0, str(OUT))
import octuple_worker
importlib.reload(octuple_worker)          # pick up edits if you re-run this cell
print("worker module ->", WORKER_PY)

BAR_IDX = FIELDS["Bar"]
BAR0_ID = VOCABS[BAR_IDX]["Bar_0"]
print(f"Bar field index {BAR_IDX} | id of Bar_0 = {BAR0_ID}")

WORKER_ARGS = (str(TOKENIZER_PATH), CONFIG["MODE"],
               CONFIG["DROP_DRUMS"], CONFIG["MIN_SEQ_NOTES"],
               CONFIG["MAX_BARS_PER_PIECE"], CONFIG["MAX_NOTES_PER_PIECE"],
               BAR_IDX, BAR0_ID, VOCAB_SIZES)

worker module -> /kaggle/working/octuple_dataset/octuple_worker.py
Bar field index 2 | id of Bar_0 = 4


In [7]:
# ---- resumable chunked write ----
def existing_chunks():
    return sorted(CHUNK_DIR.glob("chunk_*.pt"))

if existing_chunks():
    print(f"found {len(existing_chunks())} existing chunks - resuming "
          f"(delete {CHUNK_DIR} to start over)")

# Deterministic partition so a resume lines up with the same chunk indices.
FPC = CONFIG["FILES_PER_CHUNK"]
batches = [files[i:i + FPC] for i in range(0, len(files), FPC)]
print(f"{len(batches)} batches of <= {FPC:,} files")

total_seqs = total_notes = 0
t_start = time.time()

# The pool is created ONCE, outside the loop. Building it per chunk re-spawned
# every worker - and re-ran init_worker, which is what was reprinting the miditok
# warning hundreds of times - for no benefit beyond a fresh set of processes.
with ProcessPoolExecutor(max_workers=CONFIG["N_WORKERS"],
                         initializer=octuple_worker.init_worker,
                         initargs=WORKER_ARGS) as ex:
    for bi, batch in enumerate(batches):
        out_path = CHUNK_DIR / f"chunk_{bi:04d}.pt"
        if out_path.exists():
            prev = torch.load(out_path, weights_only=False)
            total_seqs += len(prev); total_notes += sum(len(s) for s in prev)
            del prev
            continue

        seqs = []
        for res in tqdm(ex.map(octuple_worker.tokenize_one, batch, chunksize=8),
                        total=len(batch), desc=f"chunk {bi+1}/{len(batches)}"):
            seqs.extend(res)

        torch.save(seqs, out_path)
        total_seqs += len(seqs); total_notes += sum(len(s) for s in seqs)
        print(f"  -> {out_path.name}: {len(seqs):,} seqs | total {total_seqs:,} seqs / "
              f"{total_notes/1e6:.1f}M notes | {(time.time()-t_start)/60:.1f} min")
        del seqs

print(f"\nDONE: {total_seqs:,} sequences, {total_notes/1e6:.1f}M notes "
      f"({total_notes*len(VOCAB_SIZES)/1e6:.0f}M raw ids) "
      f"across {len(existing_chunks())} chunks")

13 batches of <= 12,000 files


chunk 1/13:   0%|          | 0/12000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/miditok/midi_tokenizer.py:1662: UserWarning: miditok: Octuple cannot tokenize entirely this file as it contains 1670 bars whereas the limit of the tokenizer is 1024. It is therefore clipped to 1024 bars.
  tokens = self._score_to_tokens(score, attribute_controls_indexes)
/usr/local/lib/python3.12/dist-packages/miditok/midi_tokenizer.py:1662: UserWarning: miditok: Octuple cannot tokenize entirely this file as it contains 1166 bars whereas the limit of the tokenizer is 1024. It is therefore clipped to 1024 bars.
  tokens = self._score_to_tokens(score, attribute_controls_indexes)


  -> chunk_0000.pt: 2,895 seqs | total 2,895 seqs / 6.8M notes | 4.0 min


chunk 2/13:   0%|          | 0/12000 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/miditok/midi_tokenizer.py:1662: UserWarning: miditok: Octuple cannot tokenize entirely this file as it contains 1846 bars whereas the limit of the tokenizer is 1024. It is therefore clipped to 1024 bars.
  tokens = self._score_to_tokens(score, attribute_controls_indexes)
/usr/local/lib/python3.12/dist-packages/miditok/midi_tokenizer.py:1662: UserWarning: miditok: Octuple cannot tokenize entirely this file as it contains 1162 bars whereas the limit of the tokenizer is 1024. It is therefore clipped to 1024 bars.
  tokens = self._score_to_tokens(score, attribute_controls_indexes)
/usr/local/lib/python3.12/dist-packages/miditok/midi_tokenizer.py:1662: UserWarning: miditok: Octuple cannot tokenize entirely this file as it contains 1152 bars whereas the limit of the tokenizer is 1024. It is therefore clipped to 1024 bars.
  tokens = self._score_to_tokens(score, attribute_controls_indexes)


  -> chunk_0011.pt: 2,786 seqs | total 35,449 seqs / 81.8M notes | 52.0 min


chunk 13/13:   0%|          | 0/6000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/miditok/midi_tokenizer.py:1662: UserWarning: miditok: Octuple cannot tokenize entirely this file as it contains 1054 bars whereas the limit of the tokenizer is 1024. It is therefore clipped to 1024 bars.
  tokens = self._score_to_tokens(score, attribute_controls_indexes)
/usr/local/lib/python3.12/dist-packages/miditok/midi_tokenizer.py:1662: UserWarning: miditok: Octuple cannot tokenize entirely this file as it contains 1979 bars whereas the limit of the tokenizer is 1024. It is therefore clipped to 1024 bars.
  tokens = self._score_to_tokens(score, attribute_controls_indexes)
/usr/local/lib/python3.12/dist-packages/miditok/midi_tokenizer.py:1662: UserWarning: miditok: Octuple cannot tokenize entirely this file as it contains 1080 bars whereas the limit of the tokenizer is 1024. It is therefore clipped to 1024 bars.
  tokens = self._score_to_tokens(score, attribute_controls_indexes)
/usr/local/lib/python3.12/dist-packages/miditok/midi_tokenizer.p

  -> chunk_0012.pt: 1,550 seqs | total 36,999 seqs / 85.5M notes | 54.2 min

DONE: 36,999 sequences, 85.5M notes (684M raw ids) across 13 chunks


## 6. Statistics and manifest

**The token-budget arithmetic is different from REMI and it is easy to get wrong.** Your training loop consumes `TOTAL_STEPS × BATCH_SIZE × ACCUM_STEPS × SEQ_LEN` *steps*, and one Octuple step is one note. So the budget to compare against `num_notes` is the note budget — not the id budget. At `7000 × 32 × 4 × 1024` that is ~0.92B notes, which is a far larger corpus requirement than 0.92B REMI tokens represented. Expect to either raise the corpus, lower `TOTAL_STEPS`, or accept more passes over the data. `passes_over_budget` below reports which situation you are in; somewhere between 0.5 and 2 is comfortable.

**`frac_ge_min`** — the share of streams surviving `MIN_SEQ_NOTES`. This filter is harsher than the REMI one: 1025 notes is a genuinely long single-instrument part, and short tracks (pads, one-note-per-bar basslines) will be culled. If this drops below ~0.4, lower `SEQ_LEN` in training rather than lowering the minimum here, since the window builder discards short sequences anyway.

**`notes_per_bar`** is a useful sanity number — 4 to 20 for most material. Much lower and drum/percussion filtering has left you with sustained pads; much higher and you are tokenizing dense multi-track files as single streams.

In [8]:
lens, bar_spans = [], []
BAR_IDX = FIELDS["Bar"]
for p in existing_chunks():
    for s in torch.load(p, weights_only=False):
        lens.append(len(s))
        col = s[:, BAR_IDX].astype("int32")
        bar_spans.append(int(col[-1] - col[0]) + 1)

lt = torch.tensor(lens, dtype=torch.float)
bs = torch.tensor(bar_spans, dtype=torch.float)

stats = {
    "num_sequences": len(lens),
    "num_notes": int(lt.sum().item()),
    "num_raw_ids": int(lt.sum().item()) * len(VOCAB_SIZES),
    "notes_min": int(lt.min()), "notes_median": int(lt.median()),
    "notes_mean": round(float(lt.mean()), 1), "notes_max": int(lt.max()),
    "bars_median": int(bs.median()), "bars_max": int(bs.max()),
    "notes_per_bar": round(float((lt / bs.clamp(min=1)).median()), 1),
    "frac_ge_min": round(float((lt >= CONFIG["MIN_SEQ_NOTES"]).float().mean()), 3),
}
BUDGET = 7000 * 32 * 4 * 1024      # TOTAL_STEPS * BATCH * ACCUM * SEQ_LEN, in NOTES
stats["passes_over_budget"] = round(stats["num_notes"] / BUDGET, 2)

manifest = {
    "tokenization": "Octuple",
    "tokenizer": CONFIG["TOKENIZER_NAME"],
    "num_fields": len(VOCAB_SIZES),
    "vocab_sizes": VOCAB_SIZES,
    "fields": FIELDS,                 # name -> column index
    "bar_zero_id": int(BAR0_ID),
    "bars_rebased": True,
    "num_files_used": len(files),
    "config": {k: str(v) for k, v in CONFIG.items()},
    "stats": stats,
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))

print(json.dumps(stats, indent=2))
print(f"\ntraining budget ~{BUDGET/1e6:.0f}M notes -> your data covers it "
      f"{stats['passes_over_budget']}x")
print("manifest ->", OUT / "manifest.json")
assert bs.max() <= CONFIG["MAX_BAR_EMBEDDING"], "a stored piece exceeds the Bar vocabulary"

{
  "num_sequences": 36999,
  "num_notes": 85456496,
  "num_raw_ids": 683651968,
  "notes_min": 1025,
  "notes_median": 1929,
  "notes_mean": 2309.7,
  "notes_max": 4096,
  "bars_median": 87,
  "bars_max": 256,
  "notes_per_bar": 21.8,
  "frac_ge_min": 1.0,
  "passes_over_budget": 0.09
}

training budget ~918M notes -> your data covers it 0.09x
manifest -> /kaggle/working/octuple_dataset/manifest.json


## 7. Sanity check: decode back to MIDI

If this writes a listenable file, the tokenizer round-trips and the chunks are trustworthy. If it errors or sounds like noise, stop here rather than burning GPU hours — a broken tokenizer trains to a perfectly respectable loss curve and generates garbage.

Two Octuple-specific things to look for beyond "does it play":

- **Note count** should be close to the row count of the array. Octuple is one row per note, so a large discrepancy means the decoder is dropping rows — usually a `Bar`/`Position` pair it considers invalid.
- **The piece should start at bar 0.** That is the re-basing working. Decode a piece from the *middle* of a chunk, not just the first one, since the first sequence of the first file is likely to have started at bar 0 anyway.

In [9]:
chunk0 = torch.load(existing_chunks()[0], weights_only=False)
arr = chunk0[len(chunk0) // 2][:2000]          # from the middle, not the start
print("piece:", arr.shape[0], "notes x", arr.shape[1], "fields")

tk = Octuple(params=str(TOKENIZER_PATH))
ts = TokSequence(ids=[[int(v) for v in row] for row in arr], are_ids_encoded=False)
try:
    score = tk.decode(ts)
except Exception:
    score = tk.decode([ts])                     # some versions want a list
score.dump_midi(str(OUT / "sanity_check.mid"))
print("notes decoded:", sum(len(t.notes) for t in score.tracks),
      "| tracks:", len(score.tracks))
print("wrote", OUT / "sanity_check.mid")

# Field-by-field readout of the first few notes, proving Bar starts at 0.
inv = [{i: t for t, i in v.items()} for v in VOCABS]
order = sorted(FIELDS.items(), key=lambda kv: kv[1])
print("\n" + " | ".join(f"{n:>12}" for n, _ in order))
for row in arr[:8]:
    print(" | ".join(f"{str(inv[i].get(int(row[i]), '?')):>12}" for _, i in order))

piece: 1137 notes x 8 fields
notes decoded: 1137 | tracks: 1
wrote /kaggle/working/octuple_dataset/sanity_check.mid

       Pitch |     Position |          Bar |     Velocity |     Duration |      Program |        Tempo |      TimeSig
    Pitch_74 |   Position_0 |        Bar_0 | Velocity_121 | Duration_0.4.8 |    Program_1 | Tempo_127.42 |  TimeSig_4/4
    Pitch_62 |   Position_0 |        Bar_0 | Velocity_111 | Duration_0.4.8 |    Program_1 | Tempo_127.42 |  TimeSig_4/4
    Pitch_69 |   Position_4 |        Bar_0 | Velocity_111 | Duration_0.2.8 |    Program_1 | Tempo_127.42 |  TimeSig_4/4
    Pitch_57 |   Position_4 |        Bar_0 | Velocity_111 | Duration_0.2.8 |    Program_1 | Tempo_127.42 |  TimeSig_4/4
    Pitch_62 |   Position_4 |        Bar_0 | Velocity_111 | Duration_0.2.8 |    Program_1 | Tempo_127.42 |  TimeSig_4/4
    Pitch_74 |   Position_6 |        Bar_0 | Velocity_100 | Duration_0.1.8 |    Program_1 | Tempo_127.42 |  TimeSig_4/4
    Pitch_62 |   Position_6 |        Bar_0 | 

## 8. Wiring into the training notebook

In `composertrainingOctuple.ipynb` set:

```python
CHUNKS_DIR     = ".../octuple_dataset/chunks"
TOKENIZER_PATH = ".../octuple_dataset/Compose_Octuple.json"
```

On Kaggle: commit this notebook, then in the training notebook use **Add Data → Notebook Output** to mount `/kaggle/working/octuple_dataset` as a read-only input. Don't re-tokenize inside the training kernel.

**Expectations to set before you read the first loss curve:**

- Losses are **not comparable to the REMI run**, and not just because the tokenization differs. The Octuple loss is a *sum of eight cross-entropies over eight small vocabularies*, whereas REMI's was one cross-entropy over a 10k vocabulary. The absolute number will look completely different. Compare the per-field NLLs across Octuple runs, and compare generations across tokenizations.
- **Bar and Position NLL should collapse to near zero early.** In REMI, low `Bar`/`Position` loss was the interesting result. Here it is table stakes — the model can see the previous note's bar index directly in the input, so predicting the next one is close to a copy operation. The fields that carry the actual musical difficulty are `Pitch` and `Duration`.
- **One window now covers far more music.** At `SEQ_LEN=1024` notes you are giving the model minutes of context instead of a handful of bars. That is the entire point of the migration, and it is also why the training notebook recommends spending your parameter budget on depth and context rather than width.